# Space X Falcon 9 First Stage Landing Prediction

## Machine Learning Prediction (Part 5)

Predict if the Falcon 9 first stage will land successfully. We create a machine learning pipeline: standardize the data, split into training/testing sets, train four classification models (Logistic Regression, SVM, Decision Tree, KNN) with GridSearchCV hyperparameter tuning, and find the best performing model.

In [ ]:
import piplite
await piplite.install(['numpy'])
await piplite.install(['pandas'])
await piplite.install(['seaborn'])
await piplite.install(['scikit-learn'])

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import preprocessing
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
def plot_confusion_matrix(y, y_predict):
    "this function plots the confusion matrix"
    from sklearn.metrics import confusion_matrix

    cm = confusion_matrix(y, y_predict)
    ax = plt.subplot()
    sns.heatmap(cm, annot=True, ax=ax)
    ax.set_xlabel('Predicted labels')
    ax.set_ylabel('True labels')
    ax.set_title('Confusion Matrix')
    ax.xaxis.set_ticklabels(['did not land', 'land'])
    ax.yaxis.set_ticklabels(['did not land', 'landed'])
    plt.show()

In [ ]:
from js import fetch
import io

URL1 = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_2.csv'
resp1 = await fetch(URL1)
text1 = io.BytesIO((await resp1.arrayBuffer()).to_py())
data = pd.read_csv(text1)
data.head()

In [ ]:
URL2 = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_3.csv'
resp2 = await fetch(URL2)
text2 = io.BytesIO((await resp2.arrayBuffer()).to_py())
X = pd.read_csv(text2)
X.head(100)

### TASK 1
Create a NumPy array from the column `Class` in `data`, by applying the method `to_numpy()`, then assign it to the variable `Y`.

In [ ]:
Y = data['Class'].to_numpy()
Y

### TASK 2
Standardize the data in `X` then reassign it to the variable `X` using the transform provided below.

In [ ]:
transform = preprocessing.StandardScaler()
X = transform.fit_transform(X)
X

### TASK 3
Use the function `train_test_split` to split the data `X` and `Y` into training and test data. Set the parameter `test_size` to 0.2 and `random_state` to 2.

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=2)
print('Train set:', X_train.shape, Y_train.shape)
print('Test set:', X_test.shape, Y_test.shape)

In [ ]:
Y_test.shape

### TASK 4
Create a logistic regression object then create a GridSearchCV object `logreg_cv` with cv=10. Fit the object to find the best parameters.

In [ ]:
parameters = {'C': [0.01, 0.1, 1], 'penalty': ['l2'], 'solver': ['lbfgs']}
lr = LogisticRegression()
logreg_cv = GridSearchCV(lr, parameters, cv=10)
logreg_cv.fit(X_train, Y_train)

In [ ]:
print('tuned hyperparameters :(best parameters) ', logreg_cv.best_params_)
print('accuracy :', logreg_cv.best_score_)

### TASK 5
Calculate the accuracy on the test data using the method `score`.

In [ ]:
logreg_test_accuracy = logreg_cv.score(X_test, Y_test)
print('Logistic Regression test accuracy:', logreg_test_accuracy)

In [ ]:
yhat = logreg_cv.predict(X_test)
plot_confusion_matrix(Y_test, yhat)

### TASK 6
Create a support vector machine object then create a GridSearchCV object `svm_cv` with cv=10. Fit the object to find the best parameters.

In [ ]:
parameters = {'kernel': ('linear', 'rbf', 'poly', 'sigmoid'),
              'C': np.logspace(-3, 3, 5),
              'gamma': np.logspace(-3, 3, 5)}
svm = SVC()
svm_cv = GridSearchCV(svm, parameters, cv=10)
svm_cv.fit(X_train, Y_train)

In [ ]:
print('tuned hyperparameters :(best parameters) ', svm_cv.best_params_)
print('accuracy :', svm_cv.best_score_)

### TASK 7
Calculate the accuracy on the test data using the method `score`.

In [ ]:
svm_test_accuracy = svm_cv.score(X_test, Y_test)
print('SVM test accuracy:', svm_test_accuracy)

In [ ]:
yhat = svm_cv.predict(X_test)
plot_confusion_matrix(Y_test, yhat)

### TASK 8
Create a decision tree classifier object then create a GridSearchCV object `tree_cv` with cv=10. Fit the object to find the best parameters.

In [ ]:
parameters = {'criterion': ['gini', 'entropy'],
              'splitter': ['best', 'random'],
              'max_depth': [2*n for n in range(1, 10)],
              'max_features': ['sqrt'],
              'min_samples_leaf': [1, 2, 4],
              'min_samples_split': [2, 5, 10]}
tree = DecisionTreeClassifier()
tree_cv = GridSearchCV(tree, parameters, cv=10)
tree_cv.fit(X_train, Y_train)

In [ ]:
print('tuned hyperparameters :(best parameters) ', tree_cv.best_params_)
print('accuracy :', tree_cv.best_score_)

### TASK 9
Calculate the accuracy of `tree_cv` on the test data using the method `score`.

In [ ]:
tree_test_accuracy = tree_cv.score(X_test, Y_test)
print('Decision Tree test accuracy:', tree_test_accuracy)

In [ ]:
yhat = tree_cv.predict(X_test)
plot_confusion_matrix(Y_test, yhat)

### TASK 10
Create a k nearest neighbors object then create a GridSearchCV object `knn_cv` with cv=10. Fit the object to find the best parameters.

In [ ]:
parameters = {'n_neighbors': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
              'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute'],
              'p': [1, 2]}
KNN = KNeighborsClassifier()
knn_cv = GridSearchCV(KNN, parameters, cv=10)
knn_cv.fit(X_train, Y_train)

In [ ]:
print('tuned hyperparameters :(best parameters) ', knn_cv.best_params_)
print('accuracy :', knn_cv.best_score_)

### TASK 11
Calculate the accuracy of `knn_cv` on the test data using the method `score`.

In [ ]:
knn_test_accuracy = knn_cv.score(X_test, Y_test)
print('KNN test accuracy:', knn_test_accuracy)

In [ ]:
yhat = knn_cv.predict(X_test)
plot_confusion_matrix(Y_test, yhat)

### TASK 12
Find the method that performs best.

In [ ]:
models = {
    'Logistic Regression': {'cv_accuracy': logreg_cv.best_score_, 'test_accuracy': logreg_cv.score(X_test, Y_test)},
    'Support Vector Machine': {'cv_accuracy': svm_cv.best_score_, 'test_accuracy': svm_cv.score(X_test, Y_test)},
    'Decision Tree': {'cv_accuracy': tree_cv.best_score_, 'test_accuracy': tree_cv.score(X_test, Y_test)},
    'K Nearest Neighbors': {'cv_accuracy': knn_cv.best_score_, 'test_accuracy': knn_cv.score(X_test, Y_test)},
}

comparison = pd.DataFrame(models).T
print(comparison)

best_model = comparison['cv_accuracy'].idxmax()
print('\nBest performing model (by cross-validated accuracy):', best_model)

In [ ]:
# Visualize the comparison of model accuracies
comparison.plot(kind='bar', figsize=(10, 6))
plt.title('Classification Model Accuracy Comparison')
plt.ylabel('Accuracy')
plt.xticks(rotation=15)
plt.ylim(0.5, 1.0)
plt.legend(['Cross-validated accuracy', 'Test accuracy'])
plt.show()

## Conclusion

All four models achieve a similar test accuracy of about **83.3%** on the 18-sample hold-out set. The **Decision Tree** classifier achieves the highest cross-validated training accuracy (~88.9%), making it the best performing model for predicting Falcon 9 first-stage landing success. The confusion matrix shows the main source of error is **false positives** — cases where the model predicts a successful landing but the stage does not land.